In [2]:
import json
import duckdb
import pandas as pd
from typing import Any, TypedDict
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage


from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

from langchain_core.tools import tool

In [3]:
DB_PATH = "salesdata.db"


## Making basic chatting agent


In [4]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
agent = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=groq_key,
        temperature=0,         
    )

In [5]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
def load_llm():
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=groq_key,
        temperature=0,         
    )

In [6]:
messages = [
        SystemMessage(content="You are a helpful assistant.")
    ]

while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ("quit", "exit", "q"):
            print("Bye.")
            break

        if not user_input:
            continue

        # append user turn
        messages.append(HumanMessage(content=user_input))

        # call LLM with full history
        response = agent.invoke(messages)

        # append LLM turn (AIMessage) so next call has context
        messages.append(response)

        print(f"\nAgent: {response.content}")


Bye.


## Defining the tools 

In [7]:
@tool
def list_tables():
    """
    Returns the names of every table in the database.
    Call this first so you know what is available.
    """
    conn = duckdb.connect(DB_PATH)
    rows = conn.execute(
        "SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'main'"
    ).fetchall()
    conn.close()
    return ", ".join(r[0] for r in rows)

@tool
def get_schema(table_name: str) -> str:
    """
    Returns the column names, types, and 3 sample rows
    for the given table. Call this before writing a query
    so you know the exact column names and data format.
    """
    conn = duckdb.connect(DB_PATH)


    cols = conn.execute(f"DESCRIBE {table_name}").fetchdf()


    sample = conn.execute(f"SELECT * FROM {table_name} LIMIT 3").fetchdf()

    conn.close()

    return (
        f"=== {table_name} columns ===\n"
        f"{cols.to_string(index=False)}\n\n"
        f"=== sample rows ===\n"
        f"{sample.to_string(index=False)}"
    )

@tool
def run_query(sql: str) -> str:
    """
    Executes a SQL SELECT query against the database
    and returns the results as a table.
    Only SELECT statements are allowed — no INSERT, UPDATE, DELETE, or DROP.
    """
    # safety guard — only SELECT
    cleaned = sql.strip().upper()
    if not cleaned.startswith("SELECT"):
        return "ERROR: Only SELECT queries are allowed."

    try:
        conn = duckdb.connect(DB_PATH)
        df = conn.execute(sql).fetchdf()
        conn.close()
        return df.to_string(index=False)
    except Exception as e:
        return f"ERROR: {e}"
    
tools = [list_tables, get_schema, run_query]


In [8]:
TOOL_MAP = {t.name : t for t in tools}
TOOL_MAP

{'list_tables': StructuredTool(name='list_tables', description='Returns the names of every table in the database.\nCall this first so you know what is available.', args_schema=<class 'langchain_core.utils.pydantic.list_tables'>, func=<function list_tables at 0x10e067ce0>),
 'get_schema': StructuredTool(name='get_schema', description='Returns the column names, types, and 3 sample rows\nfor the given table. Call this before writing a query\nso you know the exact column names and data format.', args_schema=<class 'langchain_core.utils.pydantic.get_schema'>, func=<function get_schema at 0x11ebc2f20>),
 'run_query': StructuredTool(name='run_query', description='Executes a SQL SELECT query against the database\nand returns the results as a table.\nOnly SELECT statements are allowed — no INSERT, UPDATE, DELETE, or DROP.', args_schema=<class 'langchain_core.utils.pydantic.run_query'>, func=<function run_query at 0x11ebc2480>)}

In [9]:
def handle_response(response):
    """
    No tool calls  →  (True,  [])
    Tool calls     →  (False, [ToolMessage, …])
    """
    if not response.tool_calls:
        return True, []

    tool_messages = []
    for call in response.tool_calls:
        name = call["name"]
        args = call["args"]
        tid  = call["id"]

        print(f"  [tool call]   {name}({args})")
        result = TOOL_MAP[name].invoke(args)
        print(f"  [tool result] {result[:200]}{'…' if len(result) > 200 else ''}")

        tool_messages.append(ToolMessage(content=result, tool_call_id=tid))

    return False, tool_messages



In [10]:
def build_system_prompt() -> str:
    # hit the DB once — get the real table names
    real_tables = list_tables.invoke({})

    return (
        "You are a helpful data assistant with access to a database.\n\n"

        # ── what tables exist (hard-coded, no guessing) ──
        f"The database contains these tables: {real_tables}\n"
        "ONLY use these exact table names. Never invent table names.\n\n"

        # ── how to use the tools ──
        "You have 3 tools:\n"
        "  • list_tables  – returns all table names (already done for you above)\n"
        "  • get_schema   – call this with a table_name to see its columns and sample data\n"
        "  • run_query    – call this with a SQL SELECT to get results\n\n"

        # ── the workflow the LLM should follow ──
        "Workflow:\n"
        "  1. If you don't know a table's columns, call get_schema first.\n"
        "  2. Write a SELECT query and call run_query.\n"
        "  3. Read the results and answer the user in plain language.\n\n"

        # ── error handling instruction ──
        "If run_query returns an ERROR, read the error message carefully "
        "and fix the SQL. Try again. Do not give up after one error.\n\n"

        # ── when NOT to use tools ──
        "If the user asks a general knowledge question that has nothing to do "
        "with data, just answer directly. Do not call any tools."
    )


In [11]:
# ──────────────────────────────────────────────
# 1.  Schema fetcher
#     DS-1 tables: full dump (small, has the join logic)
#     DS-2 table:  compacted (36 month columns → one summary line)
# ──────────────────────────────────────────────

def fetch_schemas() -> str:
    conn = duckdb.connect(DB_PATH)
    parts = []

    # ── DS-1: full columns + 2 sample rows ──
    for tbl in ["salesperson", "orders", "training", "bonus_pay"]:
        cols   = conn.execute(f"DESCRIBE {tbl}").fetchdf()
        sample = conn.execute(f"SELECT * FROM {tbl} LIMIT 2").fetchdf()
        parts.append(
            f"TABLE: {tbl}\n"
            f"  columns: {list(cols['column_name'])}\n"
            f"  sample:\n{sample.to_string(index=False)}\n"
        )

    # ── DS-2: compact ──
    sample = conn.execute(
        "SELECT agent_name, agent_id, upline_manager, "
        "upline_id, agency_name FROM agent_commissions LIMIT 3"
    ).fetchdf()
    parts.append(
        "TABLE: agent_commissions\n"
        "  columns: agent_name, agent_id, upline_manager, upline_id, agency_name,\n"
        "           Jan-2022, Feb-2022, … Dec-2024   (36 monthly DOUBLE columns)\n"
        "  to reference a month in SQL use quoted name: \"Jan-2022\"\n"
        "  sample (key cols):\n"
        f"{sample.to_string(index=False)}\n"
        "  WARNING: same agent_id can appear in multiple rows "
        "with different upline_manager / agency_name.\n"
    )

    conn.close()
    return "\n".join(parts)


# ──────────────────────────────────────────────
# 2.  Planner prompt
# ──────────────────────────────────────────────

PLANNER_PROMPT = """\
You are a database query planner. You do NOT write SQL.
Your job: read the schemas and the question, then produce
a precise plan that a SQL-writing agent can follow.

════════════════════════════════════════════════
SCHEMAS
════════════════════════════════════════════════
{schemas}

════════════════════════════════════════════════
BUSINESS RULES  (you MUST apply these — they define the data)
════════════════════════════════════════════════

RULE 1 — Valid / Invalid Orders
  A salesperson has one or more training windows defined in
  the "training" table (start_date … end_date).
  An order is VALID  when its order_date falls INSIDE a window
  for that salesperson.
  An order is INVALID when its order_date falls OUTSIDE every
  window for that salesperson.
  Join key: orders.salesperson_id = training.salesperson_id
  Date check: orders.order_date BETWEEN training.start_date AND training.end_date
  To find invalid orders use:
    LEFT JOIN training ON (salesperson_id match AND date BETWEEN)
    WHERE training.id IS NULL

RULE 2 — Bonus Calculation
  1. Sum only VALID order amounts per salesperson per year.
  2. Look up that year's tiers in bonus_pay.
  3. The salesperson earns the HIGHEST bonus where
     their valid_sales >= tier.
  Tiers are exclusive: only the highest qualifying tier pays out.

RULE 3 — DS-2 Duplicate Agents
  agent_commissions can have the same agent under multiple
  upline_manager / agency_name rows.
  Always check COUNT first.  If > 1 row exists, the plan must
  flag needs_clarification = true.

════════════════════════════════════════════════
USER QUESTION
════════════════════════════════════════════════
{question}

════════════════════════════════════════════════
OUTPUT — return ONLY this JSON, nothing else
════════════════════════════════════════════════
{{
  "needs_db": true | false,
  "tables": ["…"],
  "joins": ["left_table.col = right_table.col", …],
  "conditions": ["…"],
  "select_columns": ["…"],
  "logic_summary": "plain-English description of what the SQL should do",
  "needs_clarification": true | false,
  "clarification_question": "…or empty string",
  "notes": "anything else the SQL agent must know"
}}
"""


# ──────────────────────────────────────────────
# 3.  Cache + main entry point
# ──────────────────────────────────────────────

_SCHEMA_CACHE = None

def _schemas() -> str:
    global _SCHEMA_CACHE
    if _SCHEMA_CACHE is None:
        _SCHEMA_CACHE = fetch_schemas()
    return _SCHEMA_CACHE


def get_plan(question: str) -> dict:
    """
    Call the planner LLM once.  Returns a parsed plan dict.
    On JSON-parse failure returns a fallback dict with the raw
    response in 'notes' so the executor can still try.
    """
    llm    = load_llm()
    prompt = PLANNER_PROMPT.format(schemas=_schemas(), question=question)

    response = llm.invoke([HumanMessage(content=prompt)])
    raw      = response.content.strip()

    # strip markdown fences if present
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:-1])

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        return {
            "needs_db": True,
            "tables": [],
            "joins": [],
            "conditions": [],
            "select_columns": [],
            "logic_summary": "",
            "needs_clarification": False,
            "clarification_question": "",
            "notes": f"PARSE ERROR – {e}\nraw:\n{raw}",
        }


In [12]:
EXECUTOR_SYSTEM = (
    "You are a SQL-writing agent. You will be given:\n"
    "  1. A query plan (which tables, joins, conditions)\n"
    "  2. The full table schemas\n"
    "  3. The user's original question\n\n"
    "Your job:\n"
    "  - Write a SQL SELECT that follows the plan exactly.\n"
    "  - Call run_query with that SQL.\n"
    "  - If the query errors, read the error and fix it. Try again.\n"
    "  - Once you have results, answer the user in plain language.\n\n"
    "Rules:\n"
    "  - Follow the plan's joins and conditions. Do not invent logic.\n"
    "  - If the plan says needs_clarification, ask the user before querying.\n"
    "  - Only use SELECT. Never DROP, DELETE, or INSERT.\n"
)

In [ ]:
MAX_TOOL_ROUNDS = 10

def main():
    llm            = load_llm()
    llm_with_tools = llm.bind_tools(tools)
    schemas        = _schemas()                 

    print("─" * 50)
    print("  DB Agent (Planner + Executor).  Type 'quit' to exit.")
    print("─" * 50)

    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ("quit", "exit", "q"):
            print("Bye.")
            break
        if not user_input:
            continue

        # ── Step A: run the planner ──────────────
        print("\n  [planner] thinking …")
        plan = get_plan(user_input)
        print(f"  [planner] done →  tables={plan.get('tables')}")
        if plan.get("logic_summary"):
            print(f"            logic: {plan['logic_summary']}")

        # ── short-circuit: no DB needed ──────────
        if not plan.get("needs_db", True):
            # planner said this is a general question —
            # just ask the LLM directly, no tools
            response = llm.invoke([
                SystemMessage(content="You are a helpful assistant."),
                HumanMessage(content=user_input),
            ])
            print(f"\nAgent: {response.content}")
            continue

        # ── short-circuit: needs clarification ───
        if plan.get("needs_clarification"):
            print(f"\nAgent: {plan['clarification_question']}")
            continue

        # ── Step B: executor sees question + plan + schemas ──
        inp = user_input if len(conversation_history)
        executor_messages = [
            SystemMessage(content=EXECUTOR_SYSTEM),
            HumanMessage(content=(
                f"=== SCHEMAS ===\n{schemas}\n\n"
                f"=== QUERY PLAN ===\n{json.dumps(plan, indent=2)}\n\n"
                f"=== USER QUESTION ===\n{inp}\n\n"
                "Now write and run the SQL."
            )),
        ]

        # ── Step C: tool-call loop ───────────────
        for _ in range(MAX_TOOL_ROUNDS):
            response = llm_with_tools.invoke(executor_messages)
            executor_messages.append(response)

            done, tool_messages = handle_response(response)

            if done:
                print(f"\nAgent: {response.content}")
                break

            executor_messages.extend(tool_messages)
        else:
            print("\nAgent: [stopped — too many tool calls, please rephrase]")


# ── need json in scope for the f-string above ──
import json

if __name__ == "__main__":
    main()


──────────────────────────────────────────────────
  DB Agent (Planner + Executor).  Type 'quit' to exit.
──────────────────────────────────────────────────

  [planner] thinking …
  [planner] done →  tables=['salesperson', 'orders', 'training', 'bonus_pay', 'agent_commissions']
            logic: Perform column-wise analysis for all tables, including data type, string counts, and numerical distributions.
  [tool call]   run_query({'sql': "SELECT COLUMN_NAME, DATA_TYPE, NUMERIC_PRECISION, NUMERIC_SCALE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME IN ('salesperson', 'orders', 'training', 'bonus_pay', 'agent_commissions')"})
  [tool result]    column_name data_type  numeric_precision  numeric_scale
    agent_name   VARCHAR               <NA>           <NA>
      agent_id   VARCHAR               <NA>           <NA>
upline_manager   VARCHA…

Agent: Based on the provided query plan and user question, I will provide a column-wise analysis for all tables.

The analysis includes the type o